In [0]:
from pyspark.sql import functions as F
display(spark.table("neobank.metadata.pipeline_runs").orderBy(F.desc("start_time")))

In [0]:
from pyspark.sql import functions as F

b = spark.table("neobank.bronze.customers").drop("_rescued_data", "insert_timestamp")
max_id = b.agg(F.max("customer_id")).first()[0]

# 3 updates: same customer_id, new email
upd = (b.orderBy("customer_id").limit(3)
         .withColumn("email", F.concat(F.lit("updated_"), F.col("customer_id"), F.lit("@example.com"))))

# 3 new customers: copy other rows, give new ids, names and PANs
new = (b.orderBy(F.desc("customer_id")).limit(3)
         .withColumn("customer_id", F.col("customer_id") + F.lit(max_id))
         .withColumn("first_name", F.lit("Test"))
         .withColumn("last_name", F.concat(F.lit("Customer"), F.col("customer_id")))
         .withColumn("pan_number", F.concat(F.lit("TEST"), F.col("customer_id").cast("string")))
         .withColumn("email", F.concat(F.lit("test"), F.col("customer_id"), F.lit("@example.com"))))

out = upd.unionByName(new).toPandas()
path = "/Volumes/neobank/neobank/neobank/raw/customers/customers_incremental_01.csv"
out.to_csv(path, index=False)
print(len(out), "rows written")
display(out)

In [0]:
catalog = "neobank"
vol = "/Volumes/neobank/neobank/neobank"
tables = ["customers","accounts","transactions","branches","credit_bureau_reports","payment_gateway_logs"]

# 1. remove the test file (returns False if it was never created)
print(dbutils.fs.rm(f"{vol}/raw/customers/customers_incremental_01.csv"))

# 2. drop bronze and silver tables (gold is rebuilt by the job)
for t in tables:
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.{t}")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.{t}")

# 3. clear Auto Loader's memory of processed files and inferred schemas
dbutils.fs.rm(f"{vol}/_autoloader", recurse=True)

# 4. reset the audit log and watermarks
spark.sql(f"TRUNCATE TABLE {catalog}.metadata.pipeline_runs")
spark.sql(f"""UPDATE {catalog}.metadata.table_watermarks
              SET last_watermark = TIMESTAMP'1900-01-01 00:00:00', last_run_id = NULL""")